In [7]:
from jarvis.core.specie import atomic_numbers_to_symbols
import numpy as np
from jarvis.db.jsonutils import loadjson, dumpjson
from jarvis.core.composition import Composition
from tqdm import tqdm
import os
from models import load_model, get_model, get_trainer, save_model
import os
from unsloth import FastLanguageModel
from datasets import load_dataset
import pandas as pd 
from utils import eval_prompts
from sample_funcs import parse_fn
from transformers import TextStreamer
import time
import random
import numpy as np
import random


In [8]:
fourbit_model = "unsloth/mistral-7b-bnb-4bit"
path = os.path.join("./2_models_mbj_bandgap", fourbit_model.split("/")[1])
print("Models Name", fourbit_model, path)

Models Name unsloth/mistral-7b-bnb-4bit ./2_models_mbj_bandgap/mistral-7b-bnb-4bit


In [11]:
llm_model, llm_tokenizer = FastLanguageModel.from_pretrained(model_name=path)
FastLanguageModel.for_inference(llm_model)  # Enable native 2x faster inference

==((====))==  Unsloth 2025.1.5: Fast Mistral patching. Transformers: 4.46.2.
   \\   /|    GPU: Quadro RTX 8000. Max memory: 44.481 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.1.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): l

In [12]:
Z = np.arange(100) + 1
els = atomic_numbers_to_symbols(Z)
m = 1
n = 2

In [13]:
# Create an empty DataFrame with predefined columns
import pandas as pd
data = {
    'prompt': [],
    'response': [],
    'formula':[],
    'expected_prop_val': [],
    'after_alignn_prop_val': [],
    'gen_material_str': [],
    'gen_material_cif': []
}
df = pd.DataFrame(data)

In [22]:
def gen_binary_samples(llm_model, llm_tokenizer, element="B"):
    index = 0 
    for m in np.arange(1, 4):
        for n in np.arange(1, 4):
            for i in tqdm(els):
                try:
                    comp = Composition.from_dict({i: m, element: n})
                    mbj_value = random.uniform(2.5, 5)
                    prompt_example = "Below is a description of a superconductor material. Write a response that appropriately completes the request.\n\n### Instruction:\nGenerate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n### Input:\nThe chemical formula is " + comp.reduced_formula + ". The  mbj_bandgap value is "+ str(mbj_value)+ ".\n\n### Response:\n"
            
                    batch = llm_tokenizer(prompt_example, return_tensors="pt")
                    batch = {k: v.cuda() for k, v in batch.items()}
                    generate_ids = llm_model.generate(
                                    **batch,
                                    do_sample=False,
                                    max_new_tokens=4096,
                                    pad_token_id=llm_tokenizer.eos_token_id,
                                    use_cache=True)

                    gen_strs = llm_tokenizer.batch_decode(
                                    generate_ids, 
                                    skip_special_tokens=True, 
                                    clean_up_tokenization_spaces=True
                                    )
                    try:
                        material_str = gen_strs[0].replace(prompt_example, "")
                        cif_str = parse_fn(material_str)
                    except Exception as e: 
                        cif_str = None
                        print(e)
                    df.loc[index, 'prompt'] = prompt_example
                    df.loc[index, 'response'] = gen_strs
                    df.loc[index, 'formula'] = comp.reduced_formula
                    df.loc[index, 'expected_prop_val'] = mbj_value
                    df.loc[index, 'gen_material_str'] = material_str
                    df.loc[index, 'gen_material_cif'] = cif_str
                    index += 1
                    #print(df)
                    # return df
                    # print(i)

                    # print(gen_mat, len(mem))

                    # mem.append([int(m), int(n), i, gen_mat.to_dict()])

                    # dumpjson(data=mem,filename='superB.json')
                    #df.to_csv(f"./2_gen_mbj_bandgap/{fourbit_model.split('/')[1]}_generated_samples_relaxed.csv", index=False)
                except:
                    pass



In [23]:
gen_binary_samples(llm_model, llm_tokenizer)

  0%|          | 0/100 [00:00<?, ?it/s]/tmp/ipykernel_401657/3996308429.py:35: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '5.4 2.4 5.4
68 123 114
B
0.99 0.01 0.99
B
0.51 0.51 0.49
H
0.35 0.13 0.80
H
0.85 0.63 0.70' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[index, 'gen_material_str'] = material_str
/tmp/ipykernel_401657/3996308429.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '# generated using pymatgen
data_BH
_symmetry_space_group_name_H-M   'P 1'
_cell_length_a   5.40000000
_cell_length_b   2.40000000
_cell_length_c   5.40000000
_cell_angle_alpha   68.00000000
_cell_angle_beta   123.00000000
_cell_angle_gamma   114.00000000
_symmetry_Int_Tables_number   1
_chemical_formula_structural   BH
_chemical_formula_sum   'B2 H2'
_cell_volume   52.53802480
_cell_formu

,prompt,response,formula,expected_prop_val,after_alignn_prop_val,gen_material_str,gen_material_cif
0,Below is a description of a superconductor mat...,[Below is a description of a superconductor ma...,BH,3.426271,NaN,5.4 2.4 5.4\n68 123 114\nB\n0.99 0.01 0.99\nB\...,# generated using pymatgen\ndata_BH\n_symmetry...
1,Below is a description of a superconductor mat...,[Below is a description of a superconductor ma...,B2H,4.438501,NaN,NaN,NaN
2,Below is a description of a superconductor mat...,[Below is a description of a superconductor ma...,B3H,2.786737,NaN,NaN,NaN
3,Below is a description of a superconductor mat...,[Below is a description of a superconductor ma...,He2B,3.315859,NaN,NaN,NaN
4,Below is a description of a superconductor mat...,[Below is a description of a superconductor ma...,BH,3.274953,NaN,NaN,NaN
5,Below is a description of a superconductor mat...,[Below is a description of a superconductor ma...,B3H2,2.625498,NaN,NaN,NaN
6,Below is a description of a superconductor mat...,[Below is a description of a superconductor ma...,He3B,2.502156,NaN,NaN,NaN
7,Below is a description of a superconductor mat...,[Below is a description of a superconductor ma...,B2H3,3.515212,NaN,NaN,NaN
8,Below is a description of a superconductor mat...,[Below is a description of a superconductor ma...,BH,4.611933,NaN,NaN,NaN
